In [131]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [132]:
df= pd.read_csv('train.txt', sep=';', header=None, names=['text', 'emotion'])
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [133]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [134]:
df['emotion'].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [135]:
unique_emotions = df['emotion'].unique()
emotion_numbers= {emotion: i for i, emotion in enumerate(unique_emotions)}
print(emotion_numbers)

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}


lower casing

In [136]:
df['text'] = df['text'].str.lower()

removing punctuations

In [137]:
import string
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

In [138]:
df['text']= df['text'].apply(remove_punctuation)
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


removing numbers

In [139]:
def remove_numbers(text):
    for char in text:
        if char.isdigit():
            text = text.replace(char, '')
    return text

df['text']= df['text'].apply(remove_numbers)

removing emojis

In [140]:
def remove_emojis(text):
    new_text = ''
    for i in text:
        if i.isascii():
            new_text += i
    return new_text

df['text']= df['text'].apply(remove_emojis)


Removing stopwords

In [141]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
len(stop_words)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rawknee\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Rawknee\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


198

In [142]:

def remove_stopwords(text):
    word_tokens = word_tokenize(text)
    filtered_tokens = [word for word in word_tokens if word not in stop_words]
    return ' '.join(filtered_tokens)

df['text'] = df['text'].apply(remove_stopwords)
df.loc[0, 'text']


'didnt feel humiliated'

In [143]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.2, random_state=42)

Text Vectorization

In [144]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
bow = CountVectorizer()
x_train_bow = bow.fit_transform(x_train)
x_test_bow = bow.transform(x_test)

Using Naive Byes

In [145]:
from sklearn.naive_bayes import MultinomialNB         #this NB is used for text classification
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
nb_model_bow = MultinomialNB()
nb_model_bow.fit(x_train_bow, y_train)

MultinomialNB()

In [146]:

nb_predictions = nb_model_bow.predict(x_test_bow)
print(f"accuracy for bow: {accuracy_score(y_test, nb_predictions)}")


accuracy for bow: 0.7678125


In [147]:
tfidf = TfidfVectorizer()
x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)
nb_model_tfidf = MultinomialNB()
nb_model_tfidf.fit(x_train_tfidf, y_train)
nb_predictions_tfidf = nb_model_tfidf.predict(x_test_tfidf)
print(f"accuracy for tfidf: {accuracy_score(y_test, nb_predictions_tfidf)}")

accuracy for tfidf: 0.6609375


Using Logistic Regression

In [148]:
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(x_train_bow, y_train)
lr_predictions = lr_model.predict(x_test_bow)
print(f"accuracy for bow: {accuracy_score(y_test, lr_predictions)}")



lr_model_tfidf = LogisticRegression(max_iter=1000)
lr_model_tfidf.fit(x_train_tfidf, y_train)
lr_predictions_tfidf = lr_model_tfidf.predict(x_test_tfidf)
print(f"accuracy for tfidf: {accuracy_score(y_test, lr_predictions_tfidf)}")

accuracy for bow: 0.88875
accuracy for tfidf: 0.86125
